In [3]:
import itertools, time, json, numpy as np, skimage.io as io, matplotlib.pyplot as plt
from pathlib import Path
from cellpose import models, metrics
from cellpose_sam import sam_models  

ModuleNotFoundError: No module named 'skimage'

In [1]:
# cell 1 ─ imports & paths
                                  # <- part of cellpose-sam

IMG_DIR = Path("/path/to/images")        # all test images here
OUT_DIR = Path("results"); OUT_DIR.mkdir(exist_ok=True)

# helper: run one model / param set on one image
def segment(img, model, kwargs):
    masks, flows, styles = model.eval(img, **kwargs)
    return masks

# cell 2 ─ load images once into RAM
imgs = {p.stem: io.imread(p) for p in IMG_DIR.glob("*.tif")}

# cell 3 ─ define search grids
cp_grid  = list(itertools.product([None,20,40],
                                  [0.1,0.4,1.0],
                                  [-3,0,3],
                                  [0,20]))
cpsam_grid = list(itertools.product([None,20,40],
                                    [0,0.5],
                                    [0.8,0.9]))

# cell 4 ─ run Cellpose‑3/4
cp_model = models.Cellpose(gpu=True, model_type='cyto')  # or nuclei, cyto2…
for d, ft, ct, ms in cp_grid:
    kw = dict(diameter=d, flow_threshold=ft,
              cellprob_threshold=ct, min_size=ms,
              channels=[0,0])
    for name,img in imgs.items():
        m = segment(img, cp_model, kw)
        np.save(OUT_DIR/f"{name}_CP_d{d}_ft{ft}_ct{ct}_ms{ms}.npy", m)

# cell 5 ─ run Cellpose‑SAM
sam_model = sam_models.sam()          # GPU auto‑detected
for d,st,pi in cpsam_grid:
    kw = dict(diameter=d, stitch_threshold=st,
              pred_iou_thresh=pi)
    for name,img in imgs.items():
        m = segment(img, sam_model, kw)
        np.save(OUT_DIR/f"{name}_SAM_d{d}_st{st}_pi{pi}.npy", m)

# cell 6 ─ evaluation helper (IoU vs GT, or simple count diff if no GT)
def iou(pred, gt):
    return metrics.mask_iou(gt, pred)   # needs cellpose>=4


ModuleNotFoundError: No module named 'skimage'